<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Latency and cost budgets

**[Latency and cost budgets](https://learning.nextia-ai.com/courses/search/m08/latency-and-cost-budgets/)** · Search and Ranking: Find the Best Match · Module 8, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** time each search stage on your computer, turn milliseconds into a cost per 1,000 queries, and measure what a cache of common queries would answer.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 40 MB). No account, no key and no model download. |
| **You should know** | The search page, from [New products and the cold start](https://learning.nextia-ai.com/courses/search/m07/new-products-and-the-cold-start/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m08/latency-and-cost-budgets/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M08-L01-latency-and-cost-budgets/latency-and-cost-budgets-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, bm25s 0.3.16, PyStemmer 3.1.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2 bm25s==0.3.16 PyStemmer==3.1.0
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'bm25s', 'PyStemmer']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 41 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/catalogue.csv.gz": "569c1c4c41403d59492e4cb22be61a7e634b052f67b4d32d12a9f14e77c08c52",
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/log/searches.csv.gz": "73da1fb11cebb3891cabca5ab320be2147d1855b9f8e53890007b19fb78862ea",
    "data/embeddings/products.npy": "f422870c4ab75364fad883fa8858bd7f31f2591bad07753a8f713a0872824ab0",
    "data/embeddings/queries.npy": "5333da6bd9ac44ec1bf92ddf916ec60cb557466b90b5075711f59c8e61f96253",
    "data/costs.json": "5fa6084e2cf24ecd4d273ac62dd469eab84962b7edd7a873a11879df9cf351dc",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Time each stage

Amira's budget is 200 ms for one search. Before you choose a design, measure each stage on the computer that will run it. Run the next cell: BM25 and the dense matrix search, 50 test queries each (the query embeddings are shipped, so encoding is not timed here; Module 8 in the project times it).

In [ ]:
import time, bm25s, re
cat = catalogue()
q = queries()
tok = lambda t: [w for w in re.split(r"[^\w]+", str(t).lower()) if w]
r = bm25s.BM25(k1=1.2, b=0.75, method="lucene")
r.index([tok(t + " " + d) for t, d in zip(cat["title"], cat["description"])], show_progress=False)
P = np.load("data/embeddings/products.npy").astype(np.float32)
Q = np.load("data/embeddings/queries.npy").astype(np.float32)
test = q[(q["split"] == "test") & (q["source"] == "wands")].head(50)
times = {"bm25": [], "dense search": []}
for row, text in zip(test.index, test["query"]):
    t0 = time.perf_counter(); s = r.get_scores([w for w in tok(text) if w in r.vocab_dict]); np.argsort(-s)[:100]
    times["bm25"].append((time.perf_counter() - t0) * 1000)
    t0 = time.perf_counter(); s = P @ Q[row]; np.argpartition(-s, 100)[:100]
    times["dense search"].append((time.perf_counter() - t0) * 1000)
for k, v in times.items():
    print(f"{k:14} p50 {np.percentile(v, 50):6.2f} ms   p95 {np.percentile(v, 95):6.2f} ms")

> **Check.** Your numbers depend on your computer. Look at p95 as well as p50: the budget is for the slow searches too.

## 2. From milliseconds to money

A cloud server costs money per hour, whether it is busy or not. If one query keeps one CPU busy for `ms` milliseconds, the cost is the CPU-seconds times the price of one CPU-second. The price is an **assumption** (in `data/costs.json`); check your provider's page.

In [ ]:
costs = json.load(open("data/costs.json"))
price = costs["cpu_server_usd_per_hour"]["value"]      # an assumption: 4 vCPUs, on demand
def usd_per_1000(ms, threads=1, vcpus=4):
    """CPU-seconds per query x the price of one vCPU-second, for 1,000 queries."""
    return 1000 * (ms / 1000) * threads * price / vcpus / 3600
print(f"{price} US$ an hour for 4 vCPUs")
for ms in (5, 50, 700):
    print(f"{ms:4} ms on one thread: US${usd_per_1000(ms):.5f} per 1,000 queries")
cost_700_4 = usd_per_1000(700, threads=4)

In [ ]:
expect('cost_700_4', cost_700_4, 0.033056, tolerance=0.001)

> **Check.** A 700 ms search on 4 threads (about what the cross-encoder of Module 6 needs for 50 candidates) costs US$0.033056 per 1,000 queries: still cheap, but 700 ms is far over the 200 ms budget.

## 3. Your turn: what would a cache answer?

A **cache** keeps the results of common queries, so they need no search at all. Compute the share of the log's searches that a cache of the 100 most common query texts would answer.

In [ ]:
log = pd.read_csv("data/log/searches.csv.gz", keep_default_na=False)
counts = log["query"].value_counts()
cache_100 = ...   # the share of searches that a cache of the 100 most common query texts would answer
print(cache_100)

In [ ]:
expect('cache_100', cache_100, 0.511, tolerance=0.001)

> **Check.** 0.511 of the searches (in our synthetic log). A cache helps the head of the log; the long tail still needs a fast search.

In the project: `search bench --queries 30` times every stage, with the live models.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Test changes online](https://learning.nextia-ai.com/courses/search/m08/test-changes-online/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m08/latency-and-cost-budgets/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.